# De-AntiFake (PhonePuRe) 일괄 정화 — Enkidu 평가셋용

공식 저장소: https://github.com/cyberrrange/De-AntiFake

보호본(D2)을 읽어 정화본(D3)을 **같은 화자 폴더·같은 파일 이름**으로 만든다.

```
D2/p1089/libri_p1089-0042.wav  ─►  De-AntiFake  ─►  D3/p1089/libri_p1089-0042.wav
```

## 원본 가이드에서 바꾼 것
- **화자별 하위 폴더 지원.** D2가 `D2/<화자>/<파일>.wav` 구조다. 평탄한 폴더를 가정하던 코드를 상대 경로(`p1089/libri_p1089-0042`) 기준으로 바꿨다.
- **Whisper 대신 LibriSpeech 원본 전사 사용.** D1에 `.txt`가 이미 있으므로 받아쓰기가 필요 없다. 전사 오류 위험이 사라지고 시간도 줄어든다. 전사가 없는 파일만 Whisper로 넘어간다.
- **파일명 변환 규칙 단순화.** De-AntiFake는 `이름_p화자-챕터-문장` 형식(`-`로 나뉜 숫자 3덩어리)만 읽는다. 우리 이름 `libri_p1089-0042`는 2덩어리라 챕터 자리에 `1`을 끼워 `libri_p1089-1-0042`로 바꾼다. 되돌리기가 자명하다.
- **D4 전용 셀 제거.** 대조군은 11단계에서 `IN_DIR`/`OUT_DIR`만 바꿔 다시 돌리면 된다.

## 준비물 (`내 드라이브/xtts_exp/`)
```
D1/<화자>/<파일>.wav  원본
D1/<화자>/<파일>.txt  LibriSpeech 전사   <- 이게 있어야 Whisper를 건너뛴다
D2/<화자>/<파일>.wav  Enkidu 보호본
```

## 실행 순서
1~~10단계는 설치다.
conda 환경 두 개를 만드느라 20~30분 걸리고 1회성이다.

**처음에는 11단계에서 `N_ITEMS = 2`로 두고 완주해 시간을 재라.** 파일당 25~30초로 알려져 있으니 63개면 30분 안팎이다. 실측이 나오면 `None`으로 바꿔 전체를 돌린다.

## 1단계. 코드 내려받기

In [ ]:
!git clone https://github.com/cyberrrange/De-AntiFake.git
%cd /content/De-AntiFake

## 2단계. 코랩에 conda 설치
실행이 끝나면 **런타임이 자동으로 재시작**됩니다. 정상이에요. 재시작 후에는 이 칸을 다시 실행하지 말고 3단계부터 이어서 하세요.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 3단계. (재시작 후) conda 확인

In [ ]:
%cd /content/De-AntiFake
!conda --version

## 4단계. 모델 실행 환경(phonepure) 만들기 (10~20분)

In [ ]:
%cd /content/De-AntiFake

# 중국 미러 채널은 코랩에서 멎는 일이 잦아 뺀다 (패키지 목록은 그대로)
y = open('environment.yaml').read()
y2 = '\n'.join(l for l in y.splitlines() if 'tuna.tsinghua' not in l)
open('environment_colab.yaml', 'w').write(y2)

!conda env create -f environment_colab.yaml 2>&1 | tee /content/envcreate.log | tail -50
print("=" * 50)
!ls /usr/local/envs

In [ ]:
%cd /content/De-AntiFake
!conda run -n phonepure --no-capture-output bash install_requirements.sh 2>&1 | tail -40

## 5단계. CUDA 11.6 컴파일 도구 설치
코랩의 CUDA 13.0과 이 프로젝트(CUDA 11.6용)가 맞지 않아서, phonepure 환경 안에 11.6 도구를 따로 설치합니다.

In [ ]:
!conda install -y -n phonepure -c "nvidia/label/cuda-11.6.2" cuda-nvcc cuda-cudart-dev cuda-libraries-dev
!/usr/local/envs/phonepure/bin/nvcc --version

## 6단계. 발음 정렬 도구(MFA) 환경 만들기
MFA가 실행되려면 라이브러리 버전을 낮춰야 합니다 (`joblib<1.5`, `setuptools<81`).
환경 이름은 반드시 `aligner`여야 합니다 (스크립트가 이 이름을 사용).

In [ ]:
!conda create -y -n aligner -c conda-forge montreal-forced-aligner=3.2.1
!conda install -y -n aligner -c conda-forge "joblib<1.5"
!conda run -n aligner pip install "setuptools<81"

## 7단계. 환경변수 설정 (런타임을 재시작했다면 이 칸을 다시 실행)

In [ ]:
import os
os.environ["CUDA_HOME"] = "/usr/local/envs/phonepure"
os.environ["PATH"] = "/usr/local/envs/phonepure/bin:" + os.environ["PATH"]
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"   # 코랩 T4 GPU
os.environ["MPLBACKEND"] = "agg"             # MFA의 matplotlib 오류 방지

## 8단계. MFA 확인 및 발음 사전/모델 내려받기

In [ ]:
!conda run -n aligner --no-capture-output mfa version
!conda run -n aligner --no-capture-output mfa model download acoustic english_us_arpa
!conda run -n aligner --no-capture-output mfa model download dictionary english_us_arpa

## 9단계. 학습된 모델(체크포인트) 내려받기

In [ ]:
%cd /content/De-AntiFake
!conda run -n phonepure --no-capture-output python download_models.py 2>&1 | tail -30
!ls -la checkpoints

`purification.pkl`과 `refinement.ckpt` 두 파일이 보이면 성공입니다.

## 10단계. 실행 스크립트 설정
- `DEANTIFAKE_ROOT`를 코랩 경로로 지정
- `GPU_ID`를 0으로 지정 (코랩에는 GPU가 0번 하나뿐)

In [ ]:
!sed -i 's|^DEANTIFAKE_ROOT=.*|DEANTIFAKE_ROOT=/content/De-AntiFake|' PhonePuRe/test_PhonePuRe.sh
!sed -i 's/^GPU_ID=.*/GPU_ID=0/' PhonePuRe/test_PhonePuRe.sh
!grep -n "^DEANTIFAKE_ROOT\|^GPU_ID" PhonePuRe/test_PhonePuRe.sh

## 11단계. 드라이브 연결과 설정

- 이미 마운트됐다는 메시지가 나오면 `drive.mount("/content/drive", force_remount=True)`로 바꿔 실행한다.
- **대조군(원본을 정화만 한 것)** 이 필요하면 `IN_DIR`을 `D1`, `OUT_DIR`을 `D4`로 바꿔 11단계부터 다시 돌린다.
- **보호 기법이 여럿이면** `D2_safespeech` → `D3_safespeech` 식으로 두 줄만 바꿔 반복한다.

In [ ]:
import os, re
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

DRIVE_ROOT = "/content/drive/MyDrive/xtts_exp"
IN_DIR     = f"{DRIVE_ROOT}/D2"       # 정화할 보호본 (화자별 하위 폴더)
OUT_DIR    = f"{DRIVE_ROOT}/D3"       # 정화 결과 (같은 구조로 저장)
CLEAN_DIR  = f"{DRIVE_ROOT}/D1"       # 원본 + LibriSpeech 전사(.txt)
TEXT_CACHE = f"{DRIVE_ROOT}/texts"    # 전사 모음. 실험마다 다른 폴더를 쓸 것 (재사용 시 다른 실험 전사가 섞임)
LOG_DIR    = f"{DRIVE_ROOT}/deantifake_logs"

N_ITEMS = None          # 먼저 2개로 시간 측정 -> 괜찮으면 None
MAX_SEC = None        # 정화 길이 제한이 필요할 때만
WHISPER_MODEL = "small"   # D1 전사가 다 있으면 쓰이지 않음

for d in (OUT_DIR, TEXT_CACHE, LOG_DIR):
    os.makedirs(d, exist_ok=True)

print("입력:", IN_DIR, "| 존재:", os.path.isdir(IN_DIR))
print("원본:", CLEAN_DIR, "| 존재:", os.path.isdir(CLEAN_DIR))
print("출력:", OUT_DIR)

**대조군 D4를 만들 때만** 아래 칸을 실행한다 (원본 D1을 정화). 보호본을 정화할 때는 건너뛴다.

In [ ]:
IN_DIR     = f"{DRIVE_ROOT}/D1"
OUT_DIR    = f"{DRIVE_ROOT}/D4_new"
TEXT_CACHE = f"{DRIVE_ROOT}/texts_D4new"
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(TEXT_CACHE, exist_ok=True)

## 12단계. 입력 파일 확인

`D2/<화자>/<파일>.wav`를 순회해 `p1089/libri_p1089-0042` 형태의 상대 경로로 모은다.
이 상대 경로가 끝까지 식별자로 쓰이므로 D3도 같은 구조로 저장된다.

In [ ]:
from pathlib import Path

EXTS = (".wav", ".mp3", ".m4a", ".mp4", ".flac", ".ogg")
PAIR = Path(f"{DRIVE_ROOT}/D2")

src_files = {}
for spk_dir in sorted(PAIR.iterdir()):
    if not spk_dir.is_dir():
        continue
    for f in sorted(spk_dir.iterdir()):
        if f.suffix.lower() in EXTS:
            rel = f"{spk_dir.name}/{f.stem}"
            o = Path(IN_DIR) / f"{rel}.wav"
            assert o.exists(), f"없음: {rel}"
            src_files[rel] = str(o)

ITEMS_ALL = sorted(src_files)
n_spk = len({s.split('/')[0] for s in ITEMS_ALL})
print("입력", len(ITEMS_ALL), "개, 화자", n_spk, "명")

ITEMS = ITEMS_ALL
if N_ITEMS:
    ITEMS = ITEMS_ALL[:N_ITEMS]
    print("[시험 모드]", len(ITEMS), "개만 처리")

for s in ITEMS[:5]:
    print(" ", s)

## 13단계. 전사 준비 — D1의 LibriSpeech 전사를 쓴다

De-AntiFake의 정제 단계는 음성마다 영어 전사가 필요하다(MFA 발음 정렬용).

LibriSpeech를 쓰고 있으므로 `D1/<화자>/<파일>.txt`에 **정답 전사**가 이미 있다. Whisper로 받아쓸 이유가 없다.
받아쓰기 오류가 정렬을 망가뜨릴 위험이 사라지고 시간도 줄어든다.

전사가 없는 파일만 다음 두 칸(Whisper)으로 넘어간다. LibriSpeech만 쓴다면 그 두 칸은 건너뛰어도 된다.

In [ ]:
%%writefile /content/transcribe.py
import sys, json, os, re
import whisper

cfg = json.load(open(sys.argv[1]))
model = whisper.load_model(cfg["model"])
os.makedirs(cfg["text_cache"], exist_ok=True)
for it in cfg["items"]:
    out = os.path.join(cfg["text_cache"], it["stem"] + ".txt")
    if os.path.exists(out):
        continue
    r = model.transcribe(it["path"], language="en")
    t = r["text"].strip().lower()
    t = re.sub(r"[^a-z0-9' ]+", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    with open(out, "w") as f:
        f.write(t)
    print(it["stem"], "->", t, flush=True)

In [ ]:
def norm_text(t):
    t = re.sub(r"[^a-z0-9' ]+", " ", t.lower())
    return re.sub(r"\s+", " ", t).strip()

def cache_path(rel):
    """p1089/libri_p1089-0042 -> texts/p1089__libri_p1089-0042.txt"""
    return Path(TEXT_CACHE) / (rel.replace("/", "__") + ".txt")

copied, reused, missing = 0, 0, []
for rel in ITEMS:
    dst = cache_path(rel)
    if dst.exists() and dst.read_text(encoding="utf-8").strip():
        reused += 1
        continue
    src_txt = Path(CLEAN_DIR) / f"{rel}.txt"          # D1의 LibriSpeech 전사
    if src_txt.exists():
        dst.write_text(norm_text(src_txt.read_text(encoding="utf-8")) + "\n",
                       encoding="utf-8")
        copied += 1
    else:
        missing.append(rel)

print(f"전사 복사 {copied}개 / 기존 재사용 {reused}개 / 누락 {len(missing)}개")

if missing:
    print()
    print(f"D1에 전사가 없는 파일 {len(missing)}개:", missing[:5])
    print("-> 아래 Whisper 칸을 돌리세요. 전부 LibriSpeech면 이 줄은 안 나옵니다.")
else:
    print("전부 LibriSpeech 원본 전사 사용 - Whisper 칸은 건너뛰세요.")

### (보조) 전사가 없는 파일만 Whisper로 받아쓰기

위에서 `누락 0개`가 나왔으면 **이 칸을 건너뛴다.**

In [ ]:
import json
if missing:
    need = []
    for rel in missing:
        clean = next((f"{CLEAN_DIR}/{rel}{e}" for e in EXTS
                      if os.path.exists(f"{CLEAN_DIR}/{rel}{e}")), None)
        need.append({"stem": rel.replace("/", "__"),
                     "path": clean or src_files[rel],
                     "from": "원본(D1)" if clean else "입력(D2)"})

    print("전사에 쓰는 음성:", {n["from"] for n in need})
    json.dump({"model": WHISPER_MODEL, "text_cache": TEXT_CACHE, "items": need},
              open("/content/asr_cfg.json", "w"), ensure_ascii=False)

    if not os.path.isdir("/usr/local/envs/asr"):
        get_ipython().system("conda create -y -n asr python=3.10")
        get_ipython().system('conda run -n asr --no-capture-output pip install -q "setuptools<81" openai-whisper')
    get_ipython().system("conda run -n asr --no-capture-output python /content/transcribe.py /content/asr_cfg.json")
else:
    print("전사 누락 없음 - 건너뜁니다.")

## 14단계. 전사 확인

전사가 틀리면 발음 정렬(MFA)이 틀어져 정제 품질이 떨어진다.

- LibriSpeech 전사는 사람이 만든 정답이라 그대로 믿어도 된다.
- Whisper로 받아쓴 항목이 있으면 눈으로 확인한다.
- 숫자가 들어 있으면 발음 사전에 없는 단어가 되므로 영어 단어로 풀어 쓴다(`3` -> `three`).
- 고칠 때는 드라이브의 `texts/<화자>__<파일>.txt`를 직접 수정한 뒤 **15단계부터** 다시 실행한다.

In [ ]:
shown = 0
for rel in ITEMS:
    t = cache_path(rel).read_text(encoding="utf-8").strip()
    flag = "   <- 숫자 포함" if re.search(r"\d", t) else ""
    if flag or shown < 8:
        print(f"{rel}: {t}{flag}")
        shown += 1
print(f"\n전사 {len(ITEMS)}개 준비됨")

## 15단계. De-AntiFake 데이터 폴더 만들기

De-AntiFake는 텍스트 폴더 경로가 코드에 고정돼 있어서, 스크립트를 고치는 대신 **예제 폴더에 우리 데이터를 넣어서** 돌린다. 예제 데이터는 한 번 백업한다.

**파일명 변환**: De-AntiFake는 `이름_p화자-챕터-문장` 형식(`-`로 나뉜 숫자 3덩어리)만 읽는다.
우리 이름 `libri_p1089-0042`는 2덩어리라 챕터 자리에 `1`을 끼워 넣는다.

```
libri_p1089-0042  ->  libri_p1089-1-0042
```

되돌릴 때 `-1-`만 빼면 되므로 매핑이 자명하다. 매핑은 `deantifake_logs/`에도 남긴다.

이전 결과와 `metadata.txt` 캐시는 실행 전에 지운다. 빈 캐시가 남아 결과가 틀어진 적이 있다.

In [ ]:
import shutil, json, subprocess

DEA  = "/content/De-AntiFake"
ROOT = f"{DEA}/data/test_set_example_protected"
assert os.path.isdir(ROOT), "예제 데이터 폴더가 없습니다. 1~10단계를 먼저 끝내세요."

BACKUP = f"{DEA}/data/_examples_backup"
if not os.path.exists(BACKUP):
    os.makedirs(BACKUP)
    for sub in ("audio", "text"):
        if os.path.isdir(f"{ROOT}/{sub}"):
            shutil.copytree(f"{ROOT}/{sub}", f"{BACKUP}/{sub}")
    print("예제 데이터 백업:", BACKUP)

for sub in ("audio", "text", "audio-mfa", "audio-textgrid",
            "audio-purified-PhonePuRe_t3_step1"):
    p = f"{ROOT}/{sub}"
    assert p.startswith(ROOT + "/")
    if os.path.isdir(p):
        shutil.rmtree(p)
os.makedirs(f"{ROOT}/audio"); os.makedirs(f"{ROOT}/text")

pat_ok  = re.compile(r"^[A-Za-z0-9]+_p\d+-\d+-\d+$")    # 이미 올바른 형식
pat_our = re.compile(r"^(.+_p\d+)-(\d+)$")              # libri_p1089-0042

mapping = {}
for i, rel in enumerate(ITEMS, 1):
    stem = rel.split("/")[-1]
    if pat_ok.match(stem):
        new = stem
    else:
        m = pat_our.match(stem)
        new = f"{m.group(1)}-1-{m.group(2)}" if m else f"exp_p{9000+i}-1-{i:04d}"
    mapping[rel] = new

assert len(set(mapping.values())) == len(ITEMS), "변환된 이름이 겹칩니다."

for rel, new in mapping.items():
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", src_files[rel],
           "-vn", "-ar", "16000", "-ac", "1"]
    if MAX_SEC is not None:
        cmd += ["-t", str(MAX_SEC)]
    cmd += [f"{ROOT}/audio/{new}.wav"]
    subprocess.run(cmd, check=True)

    Path(f"{ROOT}/text/{new}.txt").write_text(
        cache_path(rel).read_text(encoding="utf-8").strip(), encoding="utf-8")

json.dump(mapping, open(f"{LOG_DIR}/mapping_{Path(IN_DIR).name}.json", "w"),
          ensure_ascii=False, indent=1)

print("오디오:", len(os.listdir(f"{ROOT}/audio")),
      "| 텍스트:", len(os.listdir(f"{ROOT}/text")))
print("이름 변환 예:")
for rel, new in list(mapping.items())[:3]:
    print(f"  {rel}  ->  {new}")

In [ ]:
import threading, time, glob, shutil, os
from pathlib import Path

ROOT = "/content/De-AntiFake/data/test_set_example_protected"
BK   = Path(OUT_DIR + "_raw_backup")      # 정화 결과를 20초마다 드라이브로 복사
BK.mkdir(parents=True, exist_ok=True)

def _watch():
    seen = set()
    while True:
        for p in glob.glob(f"{ROOT}/audio-purified-*/*.wav"):
            if p in seen:
                continue
            try:
                shutil.copy2(p, BK / os.path.basename(p))
                seen.add(p)
            except Exception:
                pass
        time.sleep(20)

threading.Thread(target=_watch, daemon=True).start()
print("백업 감시 시작 ->", BK)

## 16단계. 정화 실행

환경변수(CUDA, MPLBACKEND)는 7단계에서 설정한 것을 쓴다. 런타임을 재시작했다면 아래 칸이 다시 설정한다.
로그는 `/content/deantifake_run.log`에 저장되고 화면에는 마지막 60줄만 보인다.

**시간**: 파일당 25~30초. `N_ITEMS = 2`면 1 ~ 2분, 63개면 30분 안팎.
먼저 2개로 재보고 전체 시간을 가늠한 뒤 `N_ITEMS = None`으로 바꾼다.

실행 중에는 칸을 멈추지 않는다.

In [ ]:
import os
p = '/usr/local/envs'
c = '/content/De-AntiFake/checkpoints'
print("conda:", os.listdir(p) if os.path.isdir(p) else "없음")
print("ckpt :", os.listdir(c) if os.path.isdir(c) else "없음")

In [ ]:
!conda run -n phonepure --no-capture-output python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

In [ ]:
import os
os.environ["CUDA_HOME"] = "/usr/local/envs/phonepure"
if "/usr/local/envs/phonepure/bin" not in os.environ["PATH"]:
    os.environ["PATH"] = "/usr/local/envs/phonepure/bin:" + os.environ["PATH"]
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
os.environ["MPLBACKEND"] = "agg"
%cd /content/De-AntiFake
!conda run -n phonepure --no-capture-output bash PhonePuRe/test_PhonePuRe.sh 2>&1 | tee /content/deantifake_run.log | tail -60

In [ ]:
import glob
from pathlib import Path

BK = Path(OUT_DIR + "_raw_backup")
loc = glob.glob('/content/De-AntiFake/data/test_set_example_protected/audio-purified-*/*.wav')

print("드라이브 백업:", len(list(BK.glob('*.wav'))), "개")
print("로컬 정화결과:", len(loc), "개")

진행 막대의 숫자(`audios`)는 코드 버그로 실제의 약 두 배로 올라갈 수 있습니다. 정상입니다. 실행 중에는 칸을 멈추지 마세요.

## 17단계. 검증 (저장 전에 꼭)
- 입력/출력 개수, 누락, 예상하지 못한 출력
- 길이(입력과 거의 같아야 함), NaN, 클리핑
- **발음 정렬 실패 항목**: `audio-textgrid/metadata.txt`에서 발음 정보가 비어 있으면 그 파일은 발음 안내 없이 처리된 것입니다. 0개가 이상적입니다.
- 로그에서 정화(`t: 3`)와 정제(`refinement.ckpt`) 단계가 모두 돌았는지

In [ ]:
import re
from pathlib import Path

ROOT = "/content/De-AntiFake/data/test_set_example_protected"

pat_ok  = re.compile(r"^[A-Za-z0-9]+_p\d+-\d+-\d+$")
pat_our = re.compile(r"^(.+_p\d+)-(\d+)$")

mapping = {}
for i, rel in enumerate(ITEMS, 1):
    stem = rel.split("/")[-1]
    if pat_ok.match(stem):
        new = stem
    else:
        m = pat_our.match(stem)
        new = f"{m.group(1)}-1-{m.group(2)}" if m else f"exp_p{9000+i}-1-{i:04d}"
    mapping[rel] = new

print("mapping:", len(mapping))
print("작업 폴더 오디오:", len(list(Path(f'{ROOT}/audio').glob('*.wav'))))
print("정화 결과:", len(list(Path(ROOT).glob('audio-purified-*/*.wav'))))

In [ ]:
import os,glob, numpy as np, soundfile as sf, pandas as pd
from IPython.display import display

outs = sorted(glob.glob(f"{ROOT}/audio-purified-*/*.wav"))
in_names  = {f"{n}.wav" for n in mapping.values()}
out_names = {os.path.basename(p) for p in outs}
print("입력:", len(in_names), "| 출력:", len(out_names))
print("누락:", sorted(in_names - out_names))
print("예상하지 못한 출력:", sorted(out_names - in_names))

rows = []
for s, new in mapping.items():
    p = next((q for q in outs if os.path.basename(q) == f"{new}.wav"), None)
    if p is None:
        continue
    y_in, sr_in = sf.read(f"{ROOT}/audio/{new}.wav")
    y, sr = sf.read(p)
    rows.append(dict(item=s, sr=sr, dur_in=round(len(y_in) / sr_in, 2), dur_out=round(len(y) / sr, 2),
                     nan=bool(np.isnan(y).any()), rms=round(float(np.sqrt(np.mean(y ** 2))), 4),
                     peak=round(float(np.abs(y).max()), 3), clip=bool(np.abs(y).max() >= 0.999)))
vt = pd.DataFrame(rows); display(vt)
if len(vt) and (vt.dur_in - vt.dur_out).abs().max() > 0.1:
    print("주의: 입력과 출력 길이가 0.1초 넘게 다른 파일이 있습니다.")

mp = f"{ROOT}/audio-textgrid/metadata.txt"
empty = []
if os.path.exists(mp):
    for line in open(mp):
        parts = line.rstrip("\n").split("|")
        if len(parts) >= 5 and not parts[2].strip():
            empty.append(parts[3])
    print("발음 정보가 비어 있는 항목:", len(empty), empty[:10])
else:
    print("metadata.txt가 없습니다(정렬 단계가 돌지 않았을 수 있음).")

log = open("/content/deantifake_run.log").read()
print("로그: 'refinement.ckpt' 로드됨 ->", "refinement.ckpt" in log, "| 't: 3' 설정 ->", "t: 3" in log)

**기대하는 결과**
- 누락 0, 예상하지 못한 출력 0
- 발음 정보가 비어 있는 항목 0 (있으면 그 파일의 전사를 고치고 15단계부터 다시)
- 길이 차이가 거의 없음, `nan`과 `clip`이 모두 False

## 18단계. 원래 이름으로 드라이브에 저장 (D3)

변환했던 이름을 되돌려 `OUT_DIR` 아래 **화자 폴더 구조 그대로** 저장한다.
D1, D2, D3의 파일 이름이 같아야 XTTS 비교에서 조건별로 짝지을 수 있다.

In [ ]:
import shutil

saved, missing_out = 0, []
for rel, new in mapping.items():
    p = next((q for q in outs if os.path.basename(q) == f"{new}.wav"), None)
    if p is None:
        missing_out.append(rel)
        continue
    dst = Path(OUT_DIR) / f"{rel}.wav"          # D3/p1089/libri_p1089-0042.wav
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(p, dst)
    saved += 1

print(f"저장 {saved}개 -> {OUT_DIR}")
if missing_out:
    print("저장하지 못한 항목:", missing_out)

for spk_dir in sorted(Path(OUT_DIR).iterdir()):
    if spk_dir.is_dir():
        print(f"  {spk_dir.name}: {len(list(spk_dir.glob('*.wav')))}개")

## 19단계. 입력과 정화본 비교 (첫 항목)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Audio, display

PICK = 0                      # 다른 파일을 보려면 인덱스를 바꾼다
s0   = ITEMS[PICK]
new0 = mapping[s0]
pin  = f"{ROOT}/audio/{new0}.wav"       # 정화 입력 (16kHz 변환본)
pout = f"{OUT_DIR}/{s0}.wav"            # 정화 결과

print("원본 식별자 :", s0)
print("내부 처리명 :", new0)
print("정화 결과   :", pout)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, (t, p) in zip(axes, (("input (protected)", pin), ("purified", pout))):
    y, sr = sf.read(p)
    ax.specgram(y, Fs=sr, NFFT=512, noverlap=384, cmap="magma")
    ax.set_title(f"{t}   {len(y)/sr:.2f}s")
plt.tight_layout(); plt.show()

print("입력(보호본)"); display(Audio(filename=pin))
print("정화본");       display(Audio(filename=pout))

## 다음 단계

이 노트북이 만든 `D3`가 XTTS 비교 노트북의 입력이다. D1(원본), D2(보호본)와 **파일 이름이 같아야** 조건별 비교가 된다.

### 대조군(D4)도 만들어야 한다

깨끗한 원본을 정화만 한 세트가 있어야, 복제 점수가 낮아진 것이 **보호가 남아서인지 정화가 음성을 망가뜨려서인지** 구분된다.

11단계에서 두 줄만 바꿔 다시 돌린다.

```python
IN_DIR  = f"{DRIVE_ROOT}/D1"
OUT_DIR = f"{DRIVE_ROOT}/D4"
```

### 보호 기법이 여러 개면

기법마다 폴더를 나누고 같은 방식으로 반복한다.

| 기법 | 입력 | 출력 |
| --- | --- | --- |
| Enkidu | `D2` | `D3` |
| SafeSpeech | `D2_safespeech` | `D3_safespeech` |
| SceneGuard | `D2_sceneguard` | `D3_sceneguard` |

## 알아둘 점

1. **De-AntiFake는 일반 소음 제거기가 아니라 보호 섭동 정화용이다.** SceneGuard의 소음은 비가청 교란이 아니라 가청 장면 소음이라 이 모델의 원래 용도와 다르다. 정화가 그 소음을 얼마나 지우는지는 가정하지 말고 17~19단계와 복제 점수로 확인한다.
2. **전사 품질이 정렬 품질을 좌우한다.** LibriSpeech 전사는 사람이 만든 정답이라 안전하다. Whisper로 받아쓴 항목이 섞여 있으면 14단계에서 확인한다.
3. 정화 설정은 `PhonePuRe/configs/config.json`의 기본값(`t: 3`)을 쓴다. 설정을 바꾼 정화(공격자 스윕)는 별도 실험이다.
4. 이 노트북은 정화본의 **복제 저항성**을 판정하지 않는다. 그것은 XTTS 비교 노트북이 한다.

## 자주 막히는 곳

| 증상 | 원인 | 해결 |
| --- | --- | --- |
| `No CUDA GPUs are available` | GPU 런타임이 아니거나 `GPU_ID`가 0이 아님 | 런타임을 T4 GPU로 바꾸고 10단계 다시 실행 |
| `does not have correct format` | 파일 이름이 `-`로 나뉜 숫자 3덩어리가 아님 | 15단계 이름 변환이 돌았는지 확인 |
| `발음 정보가 비어 있는 항목`이 0이 아님 | 전사가 틀렸거나 사전에 없는 단어 | `texts/`에서 해당 전사를 고치고 15단계부터 다시 |
| 진행 막대 숫자가 실제의 두 배 | 원본 코드 버그 | 정상. 멈추지 말 것 |
| 드라이브에 파일이 안 보임 | 동기화 지연 | 잠시 기다리거나 노트북에서 직접 확인 |